# Assignment 1 · Company brochure project

## Phase 1 — Configure the project and collect two homepages

The official assignment document defines the requirements. The instructor's starter notebook was unavailable, so this notebook adapts the local Day 5 workflow. Plausible Analytics and Fathom Analytics both offer website analytics. We test what their public homepage HTML exposes to the class scraper and keep the evidence for reuse.

In [1]:
import json
from datetime import datetime, timezone
from pathlib import Path
from time import perf_counter
from urllib.parse import urljoin, urlsplit

import requests

from config import COMPANIES, MODEL, SOURCE_CONTEXT_BUDGET
from scraper import extract_homepage, headers

# Set to True for a deliberate fresh capture. Existing runs are preserved.
FETCH_NEW = False
RUNS = Path("runs")
print(f"Configured model: {MODEL} | Source budget: {SOURCE_CONTEXT_BUDGET:,} characters")

Configured model: gemini-3.5-flash-lite | Source budget: 12,000 characters


## 1. Record the comparison dimensions

Each run saves these four dimensions before collecting source material. The budget applies to future model input. The snapshots retain the full extracted text.

In [2]:
dimensions = [
    "Products and services",
    "Target customers",
    "Pricing or trial availability",
    "Employment and career information",
]

def save_json(path, data):
    path.write_text(json.dumps(data, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")

def read_json(path):
    return json.loads(path.read_text(encoding="utf-8"))

RUNS.mkdir(exist_ok=True)
previous = sorted(RUNS.glob("phase1-*/setup.json"))
collect = FETCH_NEW or not previous
if collect:
    run_id = datetime.now(timezone.utc).strftime("phase1-%Y%m%dT%H%M%S%fZ")
    run_dir = RUNS / run_id
    run_dir.mkdir()
    save_json(run_dir / "setup.json", {
        "run_id": run_id, "companies": COMPANIES, "dimensions": dimensions,
        "model": MODEL, "source_context_budget": SOURCE_CONTEXT_BUDGET,
    })
else:
    run_dir = previous[-1].parent
    setup = read_json(run_dir / "setup.json")
    if setup["companies"] != COMPANIES or setup["dimensions"] != dimensions:
        raise ValueError("Saved inputs differ. Set FETCH_NEW to True for a new capture.")

print(run_dir, "|", "Live capture" if collect else "Reusing saved homepages")

runs\phase1-20261124T144231622541Z | Reusing saved homepages


## 2. Collect each homepage once

Use the Day 5 parser on the response bytes. Save original link targets, including duplicates, so the snapshot describes what was actually on the page. Redirects are recorded as failures to keep collection within the two configured URLs.

In [3]:
def collect_homepage(company_id, company):
    source_id = company_id + "1"
    url = company["url"]
    output_path = run_dir / f"{source_id}.json"
    source = {
        "source_id": source_id, "company_id": company_id, "url": url,
        "title": "", "retrieved_at": None, "text": "", "links": [], "status": "failed",
    }
    started = perf_counter()
    http_status = None
    error = None
    try:
        response = requests.get(url, headers=headers, timeout=20, allow_redirects=False)
        http_status = response.status_code
        if 200 <= http_status < 300:
            title, text, links = extract_homepage(response.content)
            source.update(title=title, text=text, links=links, status="ok" if text else "empty")
        else:
            error = f"HTTP {http_status}"
    except requests.RequestException as exc:
        error = type(exc).__name__

    source["retrieved_at"] = datetime.now(timezone.utc).isoformat()
    entry = {
        "run_id": run_dir.name, "action": "homepage_get", "transport": "Python requests",
        "company": company["name"],
        "source_id": source_id, "method": "GET", "url": url,
        "retrieved_at": source["retrieved_at"], "duration_seconds": round(perf_counter() - started, 3),
        "http_status": http_status, "status": source["status"], "error": error,
        "input_path": (run_dir / "setup.json").as_posix(), "output_path": output_path.as_posix(),
    }
    with (run_dir / "requests.jsonl").open("a", encoding="utf-8") as log:
        log.write(json.dumps(entry) + "\n")
    save_json(output_path, source)

if collect and not (run_dir / "requests.jsonl").exists():
    for company_id, company in COMPANIES.items():
        collect_homepage(company_id, company)

## 3. Verify the saved evidence

Read the snapshots from disk. Inspect a text excerpt and topic-related link targets for each company. A career-related link can guide later research. Its presence alone says nothing about current openings.

The supplied successful run includes browser-saved HTML for both homepages. The check below also re-parses those files and compares the title, full text and all link occurrences with the saved records.

In [4]:
log_path = run_dir / "requests.jsonl"
entries = [json.loads(line) for line in log_path.read_text(encoding="utf-8").splitlines()]
assert [entry["source_id"] for entry in entries] == [key + "1" for key in COMPANIES]
print(f"Homepage requests recorded in this run: {len(entries)}")

topics = ("feature", "product", "customer", "business", "pric", "trial", "about", "career", "jobs")
unusable = []
for entry in entries:
    source = read_json(Path(entry["output_path"]))
    assert (source["url"], source["status"]) == (entry["url"], entry["status"])
    raw_path = run_dir / (source["source_id"] + ".html")
    if raw_path.exists():
        extracted = extract_homepage(raw_path.read_text(encoding="utf-8"))
        assert extracted == (source["title"], source["text"], source["links"])
    links = source["links"]
    host = urlsplit(source["url"]).hostname.removeprefix("www.")
    examples = []
    for href in dict.fromkeys(links):
        target = urlsplit(urljoin(source["url"], href))
        if (target.hostname or "").removeprefix("www.") == host and any(
            topic in href.lower() for topic in topics
        ):
            examples.append(href)

    http = entry["http_status"] if entry["http_status"] is not None else "unreported"
    print(f"\n{entry['company']} | {entry.get('transport', 'Python requests')} | HTTP {http} | {source['status']}")
    print(f"{len(source['text']):,} body characters | {len(links)} link occurrences")
    print("Title:", source["title"])
    print("Excerpt:", " ".join(source["text"].split())[:350])
    print("Example links:", ", ".join(examples[:10]))
    if entry["error"]:
        print("Request error:", entry["error"])
    if source["status"] != "ok" or not source["text"].strip() or not examples:
        unusable.append(entry["company"])

if unusable:
    raise ValueError("Homepage evidence check failed for " + ", ".join(unusable))
print("\nBoth snapshots contain readable text and relevant links.")

Homepage requests recorded in this run: 2

Plausible Analytics | Chrome View Source | HTTP unreported | ok
8,636 body characters | 121 link occurrences
Title: Plausible Analytics | Simple, privacy-friendly Google Analytics alternative
Excerpt: Why Plausible Simple analytics dashboard Get all the important stats on one single page. No training necessary. Lightweight script Loads fast and won't affect your page speed or Core Web Vitals. A fraction of the size of Google Analytics. Privacy-friendly analytics Built with visitor privacy in mind. No cookie banner required. Open source The code 
Example links: /#pricing, /about, /features

Fathom Analytics | Chrome View Source | HTTP unreported | ok
13,285 body characters | 136 link occurrences
Title: Fathom Analytics: Privacy-First Google Analytics Alternative
Excerpt: Skip to main content Features What you can do with Fathom Understand your visitors without spying How Fathom works Our script, dashboard, email reports and more EU data isolati

# Phase 2 — Select additional links

This phase uses the saved homepage links from the Phase 1 run above. Run the cells in order. The candidate lists and exact messages are saved before any model call. A recorded attempt is never retried. The final cell only reports selection results.

In [5]:
import os
import subprocess
import sys

from config import create_client
from prompts import build_selection_messages
from utils import normalize_candidates

selection_dir = RUNS / run_dir.name.replace("phase1-", "phase2-for-", 1)
selection_dir.mkdir(exist_ok=True)
for company_id, company in COMPANIES.items():
    source_path = run_dir / f"{company_id}1.json"
    source = read_json(source_path)
    if source["status"] != "ok" or source["url"] != company["url"]:
        raise ValueError(f"Expected the saved homepage of {company['name']}.")
    candidates = normalize_candidates(company["url"], source["links"])
    inputs = {
        "company": company["name"], "homepage": company["url"],
        "source_path": source_path.as_posix(), "raw_link_count": len(source["links"]),
        "candidates": candidates, "model": MODEL, "response_format": {"type": "json_object"},
        "messages": build_selection_messages(company["name"], candidates),
    }
    folder = selection_dir / company_id
    folder.mkdir(exist_ok=True)
    path = folder / "input.json"
    if path.exists():
        if read_json(path) != inputs:
            raise ValueError(f"Saved selection inputs differ in {path}.")
    else:
        save_json(path, inputs)
    print(f"{company['name']}: {len(source['links'])} homepage links -> {len(candidates)} candidates")

Plausible Analytics: 121 homepage links -> 42 candidates
Fathom Analytics: 136 homepage links -> 57 candidates


## One selection attempt per company

Each request uses the exact messages in the company's input file. An exclusive attempt file prevents a repeated cell execution from issuing the same call. Successful HTTP responses are saved before decoding the model's answer. A transport failure has no JSON selection to validate.

In [6]:
from retry_phase2 import select_once

## Run offline tests before live selection

The tests exercise a consumer with a mock fetch function. A mixed response containing one forbidden link must deliver zero items to that consumer. Live selection requires an explicit opt-in through the environment variable `RUN_LINK_SELECTION=1`. Offline replays read the recorded outcomes.

In [7]:
tests = subprocess.run(
    [sys.executable, "-m", "unittest", "-v", "test_phase2"],
    capture_output=True, text=True,
)
report = tests.stdout + tests.stderr
print(report)
if tests.returncode:
    raise RuntimeError("Phase 2 offline tests failed.")
test_path = selection_dir / "offline_tests.txt"
if not test_path.exists():
    test_path.write_text(report, encoding="utf-8")

pending = [selection_dir / key for key in COMPANIES if not (selection_dir / key / "attempt.json").exists()]
if pending and os.getenv("RUN_LINK_SELECTION") == "1":
    with create_client() as client:
        for folder in pending:
            select_once(client, folder)

for company_id, company in COMPANIES.items():
    folder = selection_dir / company_id
    result_path = folder / "result.json"
    if not result_path.exists():
        state = "attempt recorded; outcome unavailable" if (folder / "attempt.json").exists() else "not attempted"
        print(company["name"], state)
        continue
    result = read_json(result_path)
    print(company["name"], "|", result["outcome"], "| JSON:", result["validation"])
    if result["validation"] == "valid":
        print(json.dumps(result["links"], indent=2, ensure_ascii=False))
    else:
        print(result.get("reason") or result.get("error_type"))

test_empty_and_unsupported (test_phase2.NormalizationTests.test_empty_and_unsupported) ... ok
test_external_and_misleading_authorities (test_phase2.NormalizationTests.test_external_and_misleading_authorities) ... ok
test_homepage_and_policy_pages (test_phase2.NormalizationTests.test_homepage_and_policy_pages) ... ok
test_http_https_queries_and_configured_www (test_phase2.NormalizationTests.test_http_https_queries_and_configured_www) ... ok
test_prompt_preserves_exact_candidates (test_phase2.NormalizationTests.test_prompt_preserves_exact_candidates) ... ok
test_relative_fragments_and_duplicates (test_phase2.NormalizationTests.test_relative_fragments_and_duplicates) ... ok
test_duplicate_normalized_selections (test_phase2.ValidationTests.test_duplicate_normalized_selections) ... ok
test_forbidden_item_rejects_entire_response (test_phase2.ValidationTests.test_forbidden_item_rejects_entire_response) ... ok
test_invalid_json (test_phase2.ValidationTests.test_invalid_json) ... ok
test_requir

# Phase 3 - Read the collected evidence

From the project directory, run `python collect_sources.py --execute` locally to collect the selected pages. Each explicit run writes a new folder. This cell can run on its own and reads the latest completed collection. Both audiences use the same saved company context. The comparison has its own single character budget across both companies. Inspect the allocation for any evidence gaps before requesting replacement sources.

In [ ]:
from pathlib import Path
from collect_sources import read_json

completed = sorted(Path("runs").glob("phase3-*/context_comparison.json"))
if completed:
    collection_dir = completed[-1].parent
    sources = [read_json(path) for path in sorted(collection_dir.glob("[AB][1-4].json"))]
    contexts = {name: read_json(collection_dir / f"context_{name}.json")
                for name in ("A", "B", "comparison")}
    print(collection_dir)
    for name, record in contexts.items():
        print(f"\n{name}: {record['used_chars']}/{record['budget']} source characters")
        for row in record["allocations"]:
            print(row["source_id"], row["action"], row["used_chars"], row["reason"])
else:
    print("No saved Phase 3 collection yet. Run python collect_sources.py --execute locally.")

# Phase 4 - Brochures and comparison

Run `python generate_outputs.py --execute` once locally. The initial plan has six calls. B-candidate displays its stream in the terminal. Use `--output B-candidate` with `--execute` for an explicit individual attempt. Completed outputs are reused. Each attempt stores its inputs, raw response or events, output and validation under `runs/phase4-*`. Evaluation criteria and the factual sampling rule are saved before the calls.

This standalone cell reads the stored outputs offline. The word count uses the brochure body including headings and excludes the final sources section.

In [ ]:
from pathlib import Path
from IPython.display import Markdown, display
from generate_outputs import OUTPUTS, saved_results

results = saved_results()
for name in OUTPUTS:
    result = results.get(name)
    if not result:
        print(name, "| No recorded attempt")
        continue
    print(name, "|", result["status"], "|", result["run_id"])
    path = Path(result["output_path"])
    if path.is_file():
        display(Markdown(path.read_bytes().decode("utf-8")))
    print(result.get("validation", {}))

## Phase 5 - Quality review and prompt evaluation

Run the following cells from the project directory to view the saved review offline. All editorial copies were prepared after generation. The original B-candidate stream failed the minimum length. The eight proposed claim verdicts await human approval.

In [ ]:
import csv
from pathlib import Path
from IPython.display import Markdown, display

review_dir = Path("phase5")
for report in ("quality_review.md", "prompt_experiment.md"):
    report_text = (review_dir / report).read_text(encoding="utf-8")
    report_text = report_text.replace("](../runs/", "](runs/").replace("](reviewed/", "](phase5/reviewed/")
    display(Markdown(report_text.replace("](claim_audit.csv)", "](phase5/claim_audit.csv)")))

In [ ]:
with (review_dir / "claim_audit.csv").open(encoding="utf-8-sig", newline="") as audit_file:
    for claim in csv.DictReader(audit_file):
        display(Markdown(f"### {claim['brochure']} / {claim['claim_number']}"))
        for label, field in (("Original claim", "exact_original_claim"), ("Verdict", "verdict"),
                             ("Reason", "explanation"), ("Correction", "correction"), ("Status", "review_status")):
            print(f"{label}: {claim[field]}")
        for prefix in ("", "second_"):
            if claim[prefix + "source_id"]:
                print(claim[prefix + "source_id"], claim[prefix + "source_url"])
                print(claim[prefix + "exact_evidence"])

In [ ]:
for copy_path in sorted((review_dir / "reviewed").glob("*.md")):
    display(Markdown(f"### {copy_path.stem} / edited after generation"))
    display(Markdown(copy_path.read_text(encoding="utf-8")))